# Taller N.º 10 — Apache Kafka: topics, particiones y grupos de consumidores

**Sesión 37 · Big Data 2026-II · Universidad Externado de Colombia**

**Carácter del taller:** calificable. Kafka no entra en ningún examen, así que el producto de este taller forma parte directa de la nota del componente de talleres. El formato es el de los Talleres 7 y 8: ejecución con inteligencia artificial permitida (Fase 1), corte, cuestionario sin inteligencia artificial (Fase 2) y evidencia en capturas.

**Qué cambia respecto del Taller 8**

1. **No hay portal web.** Kafka se opera con herramientas de consola, y la evidencia son las salidas que imprimen las celdas de este notebook.
2. **La identidad de la ejecución la acredita el `ClusterId`**, un identificador aleatorio que se genera al formatear el clúster propio y que todas las celdas imprimen en su encabezado. Cumple el papel que tenía el identificador `application_…` en los Talleres 7 y 8.
3. **Se usan dos terminales además del notebook** durante la Ejecución 4.
4. **La Fase 2 se responde en Microsoft Forms.** La terminal solo puede usarse para el script de consulta `~/t10/consulta.sh`.

**Lo que este taller no usa.** Kafka no necesita HDFS ni YARN: no hace falta iniciarlos, y si ya están activos no es necesario detenerlos. Para este taller basta con Jupyter. Los productores escritos en Python se trabajan en el Taller 11.

---
## Antes de empezar (1): las cuatro capturas obligatorias

Se exigen únicamente estas cuatro. Una captura adicional no otorga puntos, y la ausencia de una obligatoria deja sin respaldo las preguntas que dependan de ella.

| N.º | Ejecución | Qué debe verse |
|---|---|---|
| **C1** | E2 | Salida de la celda E2: el encabezado, la línea del topic con su `TopicId` y las tres líneas de partición con `Leader`, `Replicas` e `Isr` |
| **C2** | E3 | Salida de la celda E3: los dos bloques de offsets por partición, el que sigue a los 10 mensajes **sin clave** y el que sigue a los 60 mensajes **con clave** |
| **C3** | E4a | Salida de la celda E4a: los **dos** miembros del grupo `grupo10`, con sus columnas `HOST`, `#PARTITIONS` y `ASSIGNMENT` |
| **C4** | E5a | Salida de la celda E5a: el valor de **M** y la tabla del grupo con `CURRENT-OFFSET`, `LOG-END-OFFSET` y `LAG` por partición, con `LAG` distinto de cero |

**Criterio de suficiencia**

- En las cuatro capturas debe verse el **encabezado** que imprime la celda: etiqueta de la ejecución, fecha y hora del sistema, y `ClusterId`. Las cuatro deben mostrar **el mismo** `ClusterId`.
- Cada imagen se relaciona en su subsección C1, C2, C3 o C4 del archivo de Word, en ese orden.
- Basta con capturar la región de la pantalla que contiene la salida de la celda, siempre que el encabezado y las cifras sean legibles.

---
## Antes de empezar (2): notebook y terminales

Este notebook se ejecuta en el **nodo maestro**. En la Ejecución 4 se necesitan además dos terminales, que se abren con `Ctrl + Alt + T` dentro de la ventana del maestro:

| Terminal | Dónde | Cómo se abre |
|---|---|---|
| **A** | Maestro (`192.168.56.101`) | Abrir la terminal y no hacer nada más |
| **B** | `nodo2` (`192.168.56.102`) | Abrir la terminal y escribir `ssh nodo2` |

Como no hay portapapeles compartido, los comandos que se escriben en las terminales son cortos a propósito: la sección de preparación deja listos unos scripts en la carpeta `~/t10/` del maestro y de `nodo2`.

**No usar «Run All» (ejecutar todas las celdas).** Las celdas de la Ejecución 4 dependen de pasos que se hacen en las terminales, y deben ejecutarse una por una, en el orden indicado.

**Kafka debe quedar encendido hasta el final de la Fase 2.** No debe ejecutarse `kafka_cluster.sh detener` ni `kafka_cluster.sh reformatear`.

**No usar el script antiguo `~/reformatear_kafka.sh`.** Formatea los nodos con una configuración que no corresponde a este clúster y deja a Kafka sin poder arrancar. El único script de este taller es `~/kafka_cluster.sh`, que escribe la celda P1.

**Repetir celdas.** Las celdas de preparación, E1, E2, E4a y E4c pueden ejecutarse de nuevo sin consecuencias. E3, E4b, E5a y E5b no se repiten: la segunda vez solo avisan, porque volver a producir o leer mensajes alteraría los conteos.

**Repetir el taller desde cero** solo con autorización de la docente y antes del corte: ejecutar `bash ~/kafka_cluster.sh reformatear` en una terminal del maestro y continuar desde la celda de parámetros. Ese comando borra el topic, el grupo y las marcas del taller, y genera un `ClusterId` nuevo, así que **todas las capturas y la hoja de registro deben rehacerse**.

---
## Preparación (P1): script de arranque del clúster

La celda siguiente escribe el script `~/kafka_cluster.sh`. No es necesario leerlo completo; lo importante es qué hace al ejecutar `iniciar`:

1. Comprueba que se ejecuta en el maestro y que hay conexión SSH con `nodo2` y `nodo3`.
2. Revisa el almacenamiento de Kafka en los tres nodos. En modo KRaft, cada nodo guarda un archivo `meta.properties` con el `cluster.id` con el que se formateó. **Si falta en algún nodo, o si los nodos no coinciden, formatea los tres con un `cluster.id` nuevo**, generado al azar. Así nace el `ClusterId` propio de este taller.
3. Si los tres brokers ya están activos y responden, **no los reinicia**: solo verifica. Por eso la celda P3 puede ejecutarse de nuevo sin perder nada.
4. Si no lo están, arranca los tres brokers casi a la vez. Un broker que no logra registrarse ante el quórum de controllers en 60 segundos termina su proceso.
5. Espera a que los tres brokers respondan y verifica el quórum.

In [1]:
%%writefile /home/hadoop/kafka_cluster.sh
#!/bin/bash
# =====================================================================
# kafka_cluster.sh - Cluster Kafka 3.8.1 en modo KRaft (tres nodos)
# Big Data 2026-II - Universidad Externado de Colombia
#
# Se ejecuta SIEMPRE desde el nodo maestro, con el usuario hadoop:
#
#   bash ~/kafka_cluster.sh iniciar      Formatea solo si hace falta, arranca
#                                        los tres brokers y verifica el cluster.
#   bash ~/kafka_cluster.sh verificar    Solo informa el estado.
#   bash ~/kafka_cluster.sh detener      Detiene los tres brokers.
#   bash ~/kafka_cluster.sh reformatear  BORRA todos los topics y grupos,
#                                        formatea de nuevo y arranca.
#
# Heap de los brokers: por defecto, kafka-server-start.sh usa -Xmx1G -Xms1G.
# Para cambiarlo:
#   KAFKA_HEAP_OPTS="-Xmx512M -Xms512M" bash ~/kafka_cluster.sh iniciar
#
# Por que "iniciar" formatea cuando hace falta: en KRaft, cada nodo guarda en
# meta.properties el cluster.id con el que se formateo. Los tres nodos deben
# compartir el mismo cluster.id y cada uno debe estar formateado con su propio
# archivo de configuracion (node.id 1, 2 y 3). Si falta meta.properties en
# algun nodo, o si los nodos no coinciden, el script formatea los tres con un
# cluster.id nuevo. Si todo coincide, conserva los datos.
#
# Por que los tres brokers arrancan casi a la vez: cada nodo es broker y
# controller. Un broker que no logra registrarse ante el quorum de controllers
# en initial.broker.registration.timeout.ms (60 s por defecto) termina su
# proceso. Con tres controllers, el quorum exige al menos dos activos.
# =====================================================================

export JAVA_HOME=/opt/hadoop/jdk
export PATH="$JAVA_HOME/bin:$PATH"

KAFKA=/opt/kafka
BOOTSTRAP="192.168.56.101:9092,192.168.56.102:9092,192.168.56.103:9092"
NOMBRES=("maestro (192.168.56.101)" "nodo2   (192.168.56.102)" "nodo3   (192.168.56.103)")
HOSTS=(local nodo2 nodo3)
CONFS=(maestro-server.properties worker1-server.properties worker2-server.properties)
ESPERA_MAX=120
SSH_OPTS="-o BatchMode=yes -o ConnectTimeout=5"

# ---------------------------------------------------------------------
# Utilidades
# ---------------------------------------------------------------------

linea() { echo "--------------------------------------------------------------------"; }

# en_nodo <indice> <comando>: ejecuta el comando en el nodo indicado.
en_nodo() {
    local i=$1
    shift
    if [ "$i" -eq 0 ]; then
        bash -c "$*"
    else
        local heap=""
        [ -n "${KAFKA_HEAP_OPTS:-}" ] && heap="export KAFKA_HEAP_OPTS='${KAFKA_HEAP_OPTS}';"
        # shellcheck disable=SC2086
        ssh $SSH_OPTS "hadoop@${HOSTS[$i]}" \
            "export JAVA_HOME=$JAVA_HOME; export PATH=\$JAVA_HOME/bin:\$PATH; $heap $*"
    fi
}

conf_de() { echo "$KAFKA/config/kraft/${CONFS[$1]}"; }

# Valor de una propiedad en el archivo de configuracion del nodo.
propiedad() {
    en_nodo "$1" "grep -E '^$2=' $(conf_de "$1") 2>/dev/null | head -1 | cut -d= -f2-"
}

# Directorios de datos del nodo: log.dirs y, si existe, metadata.log.dir.
dirs_datos() {
    local d m
    d=$(propiedad "$1" 'log\.dirs' | tr ',' ' ')
    m=$(propiedad "$1" 'metadata\.log\.dir')
    echo "$d $m" | xargs
}

# Propiedad de meta.properties (cluster.id o node.id) en el primer log.dir.
meta_de() {
    local d
    d=$(dirs_datos "$1" | awk '{print $1}')
    [ -z "$d" ] && return
    en_nodo "$1" "grep -s '^$2=' $d/meta.properties | cut -d= -f2"
}

pid_kafka() { en_nodo "$1" "pgrep -f 'kafka[.]Kafka' | head -1"; }

# Brokers que responden: se cuentan las lineas "(id: N" sin error de conexion.
# Un broker recien detenido puede seguir apareciendo en los metadatos unos
# segundos, pero con ERROR (por ejemplo, DisconnectException): no se cuenta.
brokers_registrados() {
    timeout 25 "$KAFKA/bin/kafka-broker-api-versions.sh" --bootstrap-server "$BOOTSTRAP" 2>/dev/null \
        | grep '(id: ' | grep -vc 'ERROR'
}

# Devuelve 0 si el cluster ya esta sano: tres procesos, almacenamiento
# coherente y tres brokers que responden.
cluster_sano() {
    local i
    for i in 0 1 2; do
        [ -z "$(pid_kafka "$i")" ] && return 1
    done
    almacenamiento_coherente >/dev/null || return 1
    [ "$(brokers_registrados)" -eq 3 ]
}

# ---------------------------------------------------------------------
# Comprobaciones previas
# ---------------------------------------------------------------------

comprobar_entorno() {
    local ok=0 i
    if ! hostname -I 2>/dev/null | grep -qw '192.168.56.101'; then
        echo "ERROR: este script debe ejecutarse en el nodo maestro (192.168.56.101)."
        exit 1
    fi
    for i in 1 2; do
        if ! en_nodo "$i" true 2>/dev/null; then
            echo "ERROR: no hay conexion SSH con ${HOSTS[$i]}. Verificar que la maquina virtual este encendida."
            ok=1
        fi
    done
    [ $ok -ne 0 ] && exit 1
    for i in 0 1 2; do
        if ! en_nodo "$i" "test -f $(conf_de "$i")"; then
            echo "ERROR: falta $(conf_de "$i") en ${NOMBRES[$i]}."
            ok=1
        fi
        if ! en_nodo "$i" "test -x $JAVA_HOME/bin/java"; then
            echo "ERROR: no se encuentra Java en $JAVA_HOME en ${NOMBRES[$i]}."
            ok=1
        fi
        if [ -z "$(dirs_datos "$i")" ]; then
            echo "ERROR: $(conf_de "$i") no define log.dirs en ${NOMBRES[$i]}."
            ok=1
        fi
    done
    [ $ok -ne 0 ] && exit 1
}

# ---------------------------------------------------------------------
# Detencion
# ---------------------------------------------------------------------

detener_brokers() {
    local i vivos
    for i in 0 1 2; do
        en_nodo "$i" "$KAFKA/bin/kafka-server-stop.sh >/dev/null 2>&1"
    done
    for _ in $(seq 1 15); do
        vivos=0
        for i in 0 1 2; do
            [ -n "$(pid_kafka "$i")" ] && vivos=$((vivos + 1))
        done
        [ $vivos -eq 0 ] && return 0
        sleep 2
    done
    echo "  Algunos brokers no terminaron en 30 s (puede ocurrir al detener los tres nodos a la vez,"
    echo "  porque el ultimo pierde el quorum). Se detienen de forma forzada; los mensajes ya escritos se conservan."
    for i in 0 1 2; do
        en_nodo "$i" "pkill -9 -f 'kafka[.]Kafka'"
    done
    sleep 3
}

# ---------------------------------------------------------------------
# Formato del almacenamiento
# ---------------------------------------------------------------------

# Devuelve 0 si los tres nodos estan formateados con el mismo cluster.id
# y con el node.id de su propio archivo de configuracion.
almacenamiento_coherente() {
    local i cid nid nconf primero=""
    for i in 0 1 2; do
        cid=$(meta_de "$i" cluster.id)
        nid=$(meta_de "$i" node.id)
        nconf=$(propiedad "$i" 'node\.id')
        printf "  %s  cluster.id=%-24s node.id=%s (configuracion: %s)\n" \
            "${NOMBRES[$i]}" "${cid:-SIN_FORMATO}" "${nid:--}" "$nconf"
        [ -z "$cid" ] && return 1
        [ "$nid" != "$nconf" ] && return 1
        [ -z "$primero" ] && primero=$cid
        [ "$cid" != "$primero" ] && return 1
    done
    return 0
}

formatear() {
    local uuid i dirs d
    uuid=$("$KAFKA/bin/kafka-storage.sh" random-uuid 2>/dev/null | tail -1)
    if [ -z "$uuid" ]; then
        echo "ERROR: no fue posible generar el cluster.id (kafka-storage.sh random-uuid)."
        exit 1
    fi
    echo "  cluster.id nuevo: $uuid"
    if ls /home/hadoop/t10/*.hecho >/dev/null 2>&1; then
        rm -f /home/hadoop/t10/*.hecho
        echo "  Se eliminaron las marcas del Taller 10 (~/t10/*.hecho): el notebook puede ejecutarse de nuevo desde E1."
    fi
    for i in 0 1 2; do
        dirs=$(dirs_datos "$i")
        for d in $dirs; do
            case "$d" in
                *kafka*) ;;
                *) echo "ERROR: se rehusa borrar '$d' en ${NOMBRES[$i]} (la ruta no contiene 'kafka')."; exit 1 ;;
            esac
        done
        echo "  Formateando ${NOMBRES[$i]} con $(basename "$(conf_de "$i")")..."
        if ! en_nodo "$i" "set -o pipefail; for d in $dirs; do mkdir -p \$d && find \$d -mindepth 1 -delete; done && \
                $KAFKA/bin/kafka-storage.sh format -t $uuid -c $(conf_de "$i") 2>&1 | sed 's/^/    /'"; then
            echo "ERROR: fallo el formato en ${NOMBRES[$i]}."
            exit 1
        fi
    done
}

# ---------------------------------------------------------------------
# Arranque y espera
# ---------------------------------------------------------------------

arrancar_y_esperar() {
    local i t n pid ok_seguidas=0
    echo "Arrancando los tres brokers (nodo2, nodo3 y maestro, sin pausas)..."
    for i in 1 2 0; do
        en_nodo "$i" "cd $KAFKA && bin/kafka-server-start.sh -daemon $(conf_de "$i")"
    done
    echo
    echo "Esperando a que el cluster se forme (maximo $ESPERA_MAX s)..."
    t=0
    while [ $t -lt $ESPERA_MAX ]; do
        sleep 5
        t=$((t + 5))
        for i in 0 1 2; do
            pid=$(pid_kafka "$i")
            if [ -z "$pid" ] && [ $t -ge 10 ]; then
                echo
                echo "ERROR: el broker de ${NOMBRES[$i]} se detuvo. Ultimos errores de su registro:"
                en_nodo "$i" "grep -E 'ERROR|FATAL|Exception' $KAFKA/logs/kafkaServer.out 2>/dev/null | tail -n 8" | sed 's/^/    /'
                return 1
            fi
        done
        n=$(brokers_registrados)
        echo "  ${t} s: brokers que responden ${n:-0} de 3"
        if [ "${n:-0}" -eq 3 ]; then
            ok_seguidas=$((ok_seguidas + 1))
            [ $ok_seguidas -ge 2 ] && return 0
        else
            ok_seguidas=0
        fi
    done
    echo "ERROR: el cluster no se formo en $ESPERA_MAX s."
    return 1
}

# ---------------------------------------------------------------------
# Verificacion
# ---------------------------------------------------------------------

verificar() {
    local i pid vivos=0 q lider votantes n cids cid servicios resultado=0
    linea
    echo "VERIFICACION DEL CLUSTER KAFKA    $(date '+%Y-%m-%d %H:%M:%S')"
    linea
    echo "Procesos Kafka:"
    for i in 0 1 2; do
        pid=$(pid_kafka "$i")
        if [ -n "$pid" ]; then
            echo "  ${NOMBRES[$i]}  activo (PID $pid)"
            vivos=$((vivos + 1))
        else
            echo "  ${NOMBRES[$i]}  DETENIDO"
        fi
    done

    echo
    echo "cluster.id en meta.properties:"
    cids=""
    for i in 0 1 2; do
        cid=$(meta_de "$i" cluster.id)
        echo "  ${NOMBRES[$i]}  ${cid:-SIN_FORMATO}"
        cids="$cids $cid"
    done
    if [ "$(echo "$cids" | xargs -n1 | sort -u | wc -l)" -ne 1 ]; then
        echo "  ATENCION: los tres nodos no comparten el mismo cluster.id."
        resultado=1
    fi

    echo
    echo "Quorum KRaft (kafka-metadata-quorum.sh describe --status):"
    q=$(timeout 25 "$KAFKA/bin/kafka-metadata-quorum.sh" --bootstrap-server "$BOOTSTRAP" describe --status 2>&1)
    echo "$q" | sed 's/^/  /'
    lider=$(echo "$q" | awk '/^LeaderId:/{print $2}')
    votantes=$(echo "$q" | awk '/^CurrentVoters:/{print $2}' | tr -d '[]' | tr ',' '\n' | sort -n | xargs)

    echo
    echo "Brokers registrados:"
    timeout 25 "$KAFKA/bin/kafka-broker-api-versions.sh" --bootstrap-server "$BOOTSTRAP" 2>/dev/null \
        | grep '(id: ' | sed 's/ -> ($//; s/ -> ERROR.*/  NO RESPONDE/; s/^/  /'
    n=$(brokers_registrados)

    echo
    echo "Memoria del maestro:"
    free -h | head -2 | sed 's/^/  /'
    servicios=$(jps 2>/dev/null | awk '$2 ~ /^(NameNode|SecondaryNameNode|ResourceManager|RunJar)$/ {print $2}' | xargs)
    pgrep -f 'airflow' >/dev/null 2>&1 && servicios="$servicios airflow"
    if [ -n "$(echo "$servicios" | xargs)" ]; then
        echo "  Servicios activos que este taller no usa: $(echo "$servicios" | xargs)"
        echo "  Para liberar memoria: stop-yarn.sh && stop-dfs.sh"
    fi

    echo
    linea
    if [ $vivos -eq 3 ] && [ "${n:-0}" -eq 3 ] && [ -n "$lider" ] && [ "$lider" != "-1" ] \
        && [ "$votantes" = "1 2 3" ] && [ $resultado -eq 0 ]; then
        echo "RESULTADO: CLUSTER LISTO. Tres brokers activos; lider del quorum: nodo $lider."
        linea
        return 0
    else
        echo "RESULTADO: CLUSTER NO LISTO. Revisar los mensajes anteriores."
        echo "Esperar 20 segundos y ejecutar: bash ~/kafka_cluster.sh verificar"
        echo "Si sigue sin estar listo: bash ~/kafka_cluster.sh iniciar"
        linea
        return 1
    fi
}

# ---------------------------------------------------------------------
# Programa principal
# ---------------------------------------------------------------------

case "${1:-}" in
    iniciar)
        comprobar_entorno
        if cluster_sano; then
            echo "Los tres brokers ya estan activos, responden y comparten el mismo cluster.id."
            echo "No se reinician: se conservan los topics, los grupos y el ClusterId."
            echo
            verificar
            exit $?
        fi
        echo "Deteniendo procesos Kafka previos, si los hay..."
        detener_brokers
        echo
        echo "Estado del almacenamiento:"
        if almacenamiento_coherente; then
            echo "  Los tres nodos estan formateados con el mismo cluster.id. Se conservan los topics."
        else
            echo "  Falta el formato o los nodos no coinciden. Se formatean los tres con un cluster.id nuevo."
            formatear
        fi
        echo
        arrancar_y_esperar
        echo
        verificar
        ;;
    verificar)
        comprobar_entorno
        verificar
        ;;
    detener)
        comprobar_entorno
        echo "Deteniendo los tres brokers..."
        detener_brokers
        for i in 0 1 2; do
            if [ -n "$(pid_kafka "$i")" ]; then echo "  ${NOMBRES[$i]}  sigue activo"; else echo "  ${NOMBRES[$i]}  detenido"; fi
        done
        ;;
    reformatear)
        comprobar_entorno
        echo "ATENCION: se borraran todos los topics, mensajes y grupos de consumidores."
        echo "Para cancelar, presionar Ctrl+C en los proximos 10 segundos."
        sleep 10
        detener_brokers
        formatear
        echo
        arrancar_y_esperar
        echo
        verificar
        ;;
    *)
        echo "Uso: bash ~/kafka_cluster.sh {iniciar|verificar|detener|reformatear}"
        exit 2
        ;;
esac

Writing /home/hadoop/kafka_cluster.sh


## Preparación (P2): scripts del taller

La celda siguiente crea la carpeta `~/t10/` con dos scripts y copia el primero a `nodo2`:

- `consumidor.sh`: un consumidor de consola del grupo `grupo10`, para las terminales A y B.
- `consulta.sh`: consultas de **solo lectura** sobre el topic y el grupo. Es lo único que puede ejecutarse en la terminal durante la Fase 2.

In [2]:
import os, subprocess, textwrap

T10 = "/home/hadoop/t10"
os.makedirs(T10, exist_ok=True)

CONSUMIDOR = textwrap.dedent("""\
    #!/bin/bash
    # Consumidor de consola del grupo grupo10 (Taller 10). Para detenerlo: Ctrl+C
    export JAVA_HOME=/opt/hadoop/jdk
    echo "Consumidor en $(hostname) | grupo grupo10 | topic taller10 | $(date '+%Y-%m-%d %H:%M:%S')"
    echo "Para detenerlo: Ctrl+C"
    /opt/kafka/bin/kafka-console-consumer.sh \\
      --bootstrap-server 192.168.56.101:9092,192.168.56.102:9092,192.168.56.103:9092 \\
      --topic taller10 --group grupo10 --from-beginning \\
      --property print.partition=true --property print.offset=true --property print.key=true
    """)

CONSULTA = textwrap.dedent("""\
    #!/bin/bash
    # Consultas de solo lectura del Taller 10 (permitidas en la Fase 2).
    # Uso: bash ~/t10/consulta.sh {topic|grupo|miembros|offsets|quorum}
    export JAVA_HOME=/opt/hadoop/jdk
    KB=/opt/kafka/bin
    BS=192.168.56.101:9092,192.168.56.102:9092,192.168.56.103:9092
    CID=$(grep -s '^cluster.id=' /opt/kafka/kafka-logs/meta.properties | cut -d= -f2)
    echo "======================================================================"
    echo "consulta.sh ${1:-} | $(date '+%Y-%m-%d %H:%M:%S') | ClusterId: ${CID:-NO DISPONIBLE}"
    echo "======================================================================"
    case "${1:-}" in
      topic)    $KB/kafka-topics.sh --bootstrap-server $BS --describe --topic taller10 ;;
      grupo)    $KB/kafka-consumer-groups.sh --bootstrap-server $BS --describe --group grupo10 ;;
      miembros) $KB/kafka-consumer-groups.sh --bootstrap-server $BS --describe --group grupo10 --members --verbose ;;
      offsets)  $KB/kafka-get-offsets.sh --bootstrap-server $BS --topic taller10 ;;
      quorum)   $KB/kafka-metadata-quorum.sh --bootstrap-server $BS describe --status ;;
      *)        echo "Uso: bash ~/t10/consulta.sh {topic|grupo|miembros|offsets|quorum}" ;;
    esac
    """)

for nombre, contenido in [("consumidor.sh", CONSUMIDOR), ("consulta.sh", CONSULTA)]:
    ruta = os.path.join(T10, nombre)
    with open(ruta, "w") as f:
        f.write(contenido)
    os.chmod(ruta, 0o755)
    print("Escrito:", ruta)

copia = subprocess.run(
    "ssh -o BatchMode=yes nodo2 mkdir -p /home/hadoop/t10 && "
    "scp -q /home/hadoop/t10/consumidor.sh nodo2:/home/hadoop/t10/",
    shell=True, capture_output=True, text=True)
print("Copia de consumidor.sh a nodo2:", "correcta" if copia.returncode == 0 else "FALLO -> " + copia.stderr)

Escrito: /home/hadoop/t10/consumidor.sh
Escrito: /home/hadoop/t10/consulta.sh
Copia de consumidor.sh a nodo2: correcta


## Preparación (P3): arranque del clúster Kafka

La ejecución tarda entre uno y dos minutos. La primera vez incluye el formato de los tres nodos. Debe terminar con la línea **`RESULTADO: CLUSTER LISTO`**.

Si termina con `CLUSTER NO LISTO`, esperar 20 segundos y ejecutar la celda una segunda vez: si el clúster ya se formó, la celda lo detecta y no lo reinicia. Si el resultado se repite, informar a la docente antes de continuar.

In [3]:
!bash /home/hadoop/kafka_cluster.sh iniciar

Deteniendo procesos Kafka previos, si los hay...

Estado del almacenamiento:
  maestro (192.168.56.101)  cluster.id=SIN_FORMATO              node.id=- (configuracion: 1)
  Falta el formato o los nodos no coinciden. Se formatean los tres con un cluster.id nuevo.
  cluster.id nuevo: Rxkee_6WRYaGnf3eVYjxNw
  Formateando maestro (192.168.56.101) con maestro-server.properties...
    Formatting /opt/kafka/kafka-logs with metadata.version 3.8-IV0.
  Formateando nodo2   (192.168.56.102) con worker1-server.properties...
    Formatting /opt/kafka/kafka-logs with metadata.version 3.8-IV0.
  Formateando nodo3   (192.168.56.103) con worker2-server.properties...
    Formatting /opt/kafka/kafka-logs with metadata.version 3.8-IV0.

Arrancando los tres brokers (nodo2, nodo3 y maestro, sin pausas)...

Esperando a que el cluster se forme (maximo 120 s)...
  5 s: brokers que responden 3 de 3
  10 s: brokers que responden 3 de 3

--------------------------------------------------------------------
VERIFICA

---
## Parámetros del taller

Todos los valores que gobiernan las ejecuciones están reunidos en la celda siguiente. No se modifican durante el taller.

In [4]:
# Parametros del taller. No modificar durante la sesion.
import os, random, time, datetime, subprocess

os.environ["JAVA_HOME"] = "/opt/hadoop/jdk"
os.environ["PATH"] = "/opt/hadoop/jdk/bin:" + os.environ["PATH"]

KB  = "/opt/kafka/bin"                                                # herramientas de consola
BS  = "192.168.56.101:9092,192.168.56.102:9092,192.168.56.103:9092"   # brokers de arranque
T10 = "/home/hadoop/t10"

TOPIC       = "taller10"
GRUPO       = "grupo10"
PARTICIONES = 3
REPLICACION = 2


def cluster_id():
    """Lee el cluster.id con el que se formateo el clúster propio."""
    try:
        with open("/opt/kafka/kafka-logs/meta.properties") as f:
            for linea in f:
                if linea.startswith("cluster.id="):
                    return linea.strip().split("=", 1)[1]
    except OSError:
        pass
    return "NO DISPONIBLE"


def encabezado(etiqueta):
    """Encabezado que acredita la ejecucion propia: debe verse en cada captura."""
    print("=" * 72)
    print("{} | {:%Y-%m-%d %H:%M:%S} | ClusterId: {}".format(etiqueta, datetime.datetime.now(), cluster_id()))
    print("=" * 72)


def ya_ejecutada(etiqueta):
    """Evita repetir una produccion de mensajes: alteraria los conteos del taller."""
    marca = os.path.join(T10, etiqueta + ".hecho")
    if os.path.exists(marca):
        with open(marca) as f:
            print("ATENCION: {} ya se ejecuto en este cluster ({}).".format(etiqueta, f.read().strip()))
        print("No se repite, porque hacerlo alteraria los conteos del taller.")
        return True
    return False


def marcar(etiqueta, detalle=""):
    with open(os.path.join(T10, etiqueta + ".hecho"), "w") as f:
        f.write("{:%Y-%m-%d %H:%M:%S} {}".format(datetime.datetime.now(), detalle))


def estado_grupo():
    """Devuelve (estado, miembros) del grupo. ('NO_EXISTE', 0) si aun no se ha creado."""
    r = subprocess.run([KB + "/kafka-consumer-groups.sh", "--bootstrap-server", BS,
                        "--describe", "--group", GRUPO, "--state"],
                       capture_output=True, text=True)
    salida = r.stdout + r.stderr
    if "does not exist" in salida:
        return "NO_EXISTE", 0
    for linea in salida.splitlines():
        partes = linea.split()
        if partes and partes[0] == GRUPO and partes[-1].isdigit():
            return partes[-2], int(partes[-1])
    return "DESCONOCIDO", 0


print("Parametros cargados. ClusterId:", cluster_id())

Parametros cargados. ClusterId: Rxkee_6WRYaGnf3eVYjxNw


---
## Ejecución 1 — El quórum de controllers

En modo KRaft, los metadatos del clúster (qué brokers existen, qué topics hay, dónde está cada partición) los gestiona un **quórum de controllers**. En la sala, cada nodo es a la vez broker y controller: los nodos 1, 2 y 3 son los tres votantes del quórum. Uno de ellos es el **líder** y los otros dos lo siguen, replicando el registro de metadatos.

La celda ejecuta dos consultas sobre el quórum:

- `describe --status` muestra el `ClusterId`, el `LeaderId` (nodo líder del quórum) y los votantes (`CurrentVoters`).
- `describe --replication` muestra, para cada votante, hasta dónde ha replicado el registro de metadatos (`LogEndOffset`), cuánto le falta respecto del líder (`Lag`) y su papel (`Status`: `Leader` o `Follower`).

El líder del quórum se elige entre los tres al arrancar; no tiene por qué ser el maestro.

In [5]:
encabezado("E1")
!{KB}/kafka-metadata-quorum.sh --bootstrap-server {BS} describe --status
print()
!{KB}/kafka-metadata-quorum.sh --bootstrap-server {BS} describe --replication

E1 | 2026-10-28 16:51:09 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
ClusterId:              Rxkee_6WRYaGnf3eVYjxNw
LeaderId:               2
LeaderEpoch:            1
HighWatermark:          629
MaxFollowerLag:         0
MaxFollowerLagTimeMs:   120
CurrentVoters:          [1,2,3]
CurrentObservers:       []

NodeId	LogEndOffset	Lag	LastFetchTimestamp	LastCaughtUpTimestamp	Status  	
2     	633         	0  	1793224272807     	1793224272807        	Leader  	
1     	633         	0  	1793224272551     	1793224272551        	Follower	
3     	633         	0  	1793224272552     	1793224272552        	Follower	


**Registrar en la hoja (fila E1):** `ClusterId` completo · `LeaderId` · `CurrentVoters` · nodo cuyo `Status` es `Leader` en la segunda tabla.

*No requiere captura.* El `ClusterId` registrado aquí debe coincidir con el de los encabezados de las cuatro capturas.

---
## Ejecución 2 — Un topic con tres particiones y factor de replicación 2

Se crea el topic `taller10` con 3 particiones y factor de replicación 2: cada partición tendrá dos copias, en dos brokers distintos. Kafka decide en qué brokers queda cada réplica y cuál de las dos es la **líder**; esa ubicación se elige con un punto de partida aleatorio, así que varía de un clúster a otro.

En la salida de `--describe`, la primera línea corresponde al topic e incluye su **`TopicId`**, otro identificador aleatorio. Después aparece una línea por partición:

- `Leader`: broker que atiende las escrituras y lecturas de esa partición.
- `Replicas`: brokers que guardan una copia, empezando por la réplica preferida para el liderazgo.
- `Isr`: réplicas sincronizadas con el líder, incluido él mismo.
- `Elr` y `LastKnownElr` corresponden a una función que este clúster no tiene activada; se ignoran.

La celda también imprime el valor de `min.insync.replicas` de la configuración de los brokers: el número mínimo de réplicas sincronizadas que exige una escritura con `acks=all`.

In [6]:
encabezado("E2")
!grep -E '^min.insync.replicas=' /opt/kafka/config/kraft/maestro-server.properties
print()
!{KB}/kafka-topics.sh --bootstrap-server {BS} --create --topic {TOPIC} --partitions {PARTICIONES} --replication-factor {REPLICACION}
time.sleep(3)
print()
!{KB}/kafka-topics.sh --bootstrap-server {BS} --describe --topic {TOPIC}

E2 | 2026-10-28 16:51:26 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
min.insync.replicas=2

Created topic taller10.

Topic: taller10	TopicId: 3PTJB7YzTdKMPuSJZ36N6A	PartitionCount: 3	ReplicationFactor: 2	Configs: min.insync.replicas=2,segment.bytes=536870912,retention.bytes=1073741824
	Topic: taller10	Partition: 0	Leader: 1	Replicas: 1,2	Isr: 1,2	Elr: 	LastKnownElr: 
	Topic: taller10	Partition: 1	Leader: 2	Replicas: 2,3	Isr: 2,3	Elr: 	LastKnownElr: 
	Topic: taller10	Partition: 2	Leader: 3	Replicas: 3,1	Isr: 3,1	Elr: 	LastKnownElr: 


**Si aparece el mensaje `Topic 'taller10' already exists`**, el topic se creó en una ejecución anterior de esta celda. No debe borrarse ni crearse de nuevo: basta con usar la salida de `--describe`, que es la misma.

**Registrar en la hoja (fila E2):** `TopicId` · para cada partición, `Leader`, `Replicas` e `Isr` · valor de `min.insync.replicas`.

*Razonamiento que se registra en la misma fila:* identificar el broker que aparece en `Leader` en la línea de la partición 0. Después, revisar la columna `Replicas` de las tres particiones y anotar **cuáles incluyen ese broker**; la partición 0 siempre es una de ellas. Se registra con la forma «broker X → particiones _ y _». Si ese broker se detuviera, el `Isr` de esas particiones quedaría con una sola réplica, menos que `min.insync.replicas`.

**Captura C1:** la salida completa de esta celda, con el encabezado a la vista.

---
## Ejecución 3 — ¿En qué partición escribe el productor?

Quien decide la partición de cada mensaje es el **productor**, no el broker. La celda produce dos tandas de mensajes con el productor de consola y, después de cada una, consulta el último offset de cada partición con `kafka-get-offsets.sh`. Cada línea de esa consulta tiene la forma `topic:partición:offset`; en un topic nuevo, ese offset coincide con el número de mensajes escritos en la partición.

- **E3a:** 10 mensajes **sin clave**.
- **E3b:** 60 mensajes **con clave**. Las claves tienen la forma `sensor-NNN`, con un número elegido al azar en cada ejecución, así que el reparto resultante es propio de cada clúster.

Con clave, la regla es conocida desde la lectura previa: el productor aplica una función *hash* a la clave para elegir la partición, de modo que la misma clave va siempre a la misma partición. Sin clave, conviene **observar** qué ocurre antes de explicarlo.

In [7]:
encabezado("E3")
if not ya_ejecutada("E3"):
    # E3a: diez mensajes sin clave
    with open(T10 + "/e3a.txt", "w") as f:
        for i in range(1, 11):
            f.write("lectura-sin-clave-{:02d}\n".format(i))
    !{KB}/kafka-console-producer.sh --bootstrap-server {BS} --topic {TOPIC} < {T10}/e3a.txt
    print("E3a. Offsets por particion despues de 10 mensajes SIN clave (topic:particion:offset)")
    !{KB}/kafka-get-offsets.sh --bootstrap-server {BS} --topic {TOPIC}

    # E3b: sesenta mensajes con clave aleatoria
    claves = ["sensor-{}".format(random.randint(100, 999)) for _ in range(60)]
    with open(T10 + "/e3b.txt", "w") as f:
        for i, clave in enumerate(claves, 1):
            f.write("{}:lectura-con-clave-{:02d}\n".format(clave, i))
    !{KB}/kafka-console-producer.sh --bootstrap-server {BS} --topic {TOPIC} --property parse.key=true --property key.separator=: < {T10}/e3b.txt
    print()
    print("E3b. Offsets por particion despues de 60 mensajes CON clave ({} claves distintas)".format(len(set(claves))))
    !{KB}/kafka-get-offsets.sh --bootstrap-server {BS} --topic {TOPIC}
    marcar("E3", "10 sin clave y 60 con clave")

E3 | 2026-10-28 16:58:21 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
E3a. Offsets por particion despues de 10 mensajes SIN clave (topic:particion:offset)
taller10:0:0
taller10:1:0
taller10:2:10

E3b. Offsets por particion despues de 60 mensajes CON clave (59 claves distintas)
taller10:0:16
taller10:1:26
taller10:2:28


**Registrar en la hoja (fila E3):** solo el último número de cada línea `topic:partición:offset`, en el orden partición 0 / 1 / 2. Primero los tres offsets después de E3a; después, la partición que recibió los 10 mensajes sin clave; después, los tres offsets después de E3b; y por último los mensajes con clave que recibió cada partición, que se obtienen restando, partición por partición, el bloque E3a del bloque E3b. Esas tres restas deben sumar 60.

**Captura C2:** la salida completa de esta celda, con el encabezado y los dos bloques de offsets a la vista.

*Explicación que conviene tener preparada:* por qué los diez mensajes sin clave quedaron donde quedaron. Una pista: el productor no envía cada mensaje por separado, sino que los agrupa en **lotes** por partición, y el tamaño de lote por defecto (`batch.size`) es de 16.384 bytes. Conviene comparar esa cifra con el tamaño total de los diez mensajes.

---
## Ejecución 4 — Dos consumidores en un mismo grupo

Un **grupo de consumidores** reparte las particiones del topic entre sus miembros: cada partición se asigna a **un solo** miembro del grupo. En esta ejecución se lanzan dos consumidores del grupo `grupo10`, uno en el maestro y otro en `nodo2`, y se observa cómo se reparten las tres particiones.

Seguir los pasos **en este orden**:

| Paso | Dónde | Qué hacer | Qué se espera |
|---|---|---|---|
| 1 | Terminal A | Escribir `bash ~/t10/consumidor.sh` | Aparecen los 70 mensajes de E3, uno por línea, con `Partition:`, `Offset:`, clave y valor. El grupo no tenía offsets confirmados, así que empieza desde el principio (`--from-beginning`). Los mensajes sin clave muestran `null` como clave |
| 2 | Terminal B | Escribir `ssh nodo2` y luego `bash ~/t10/consumidor.sh` | No aparece ningún mensaje: el grupo ya los leyó todos. Al entrar el segundo miembro, el grupo se **rebalancea** |
| 3 | Notebook | Esperar unos 10 segundos y ejecutar la celda **E4a** | Dos miembros, uno en cada `HOST` |
| 4 | Notebook | Ejecutar la celda **E4b** | Se producen 12 mensajes con clave. Cada terminal muestra **solo** los de sus particiones |
| 5 | Terminal B | Presionar `Ctrl + C` | El consumidor termina e imprime `Processed a total of N messages` |
| 6 | Notebook | Ejecutar la celda **E4c** | Queda un solo miembro, con las tres particiones |
| 7 | Terminal A | Presionar `Ctrl + C` | El consumidor termina e imprime su propio total |

Si la celda E4a muestra el aviso `is rebalancing` o un solo miembro, esperar 10 segundos y ejecutarla de nuevo. Si muestra `Consumer group 'grupo10' does not exist`, el consumidor de la terminal A todavía no se ha iniciado (paso 1).

In [8]:
encabezado("E4a")
estado, miembros = estado_grupo()
if estado == "NO_EXISTE":
    print("ATENCION: el grupo grupo10 todavia no existe. Iniciar el consumidor de la terminal A (paso 1).")
elif miembros < 2:
    print("ATENCION: el grupo tiene {} miembro(s) activo(s). Iniciar el consumidor de la terminal B (paso 2),".format(miembros))
    print("esperar 10 segundos y ejecutar de nuevo esta celda.")
!{KB}/kafka-consumer-groups.sh --bootstrap-server {BS} --describe --group {GRUPO} --members --verbose

E4a | 2026-10-28 17:09:13 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw

GROUP           CONSUMER-ID                                           HOST            CLIENT-ID        #PARTITIONS     ASSIGNMENT
grupo10         console-consumer-26cc7dc4-1b1f-4188-9512-91ca60ac4938 /192.168.56.102 console-consumer 1               (2)
grupo10         console-consumer-05194e76-9a68-434d-8dc0-814aab3d94db /192.168.56.101 console-consumer 2               (0,1)


**Captura C3:** la salida de la celda E4a, con los dos miembros visibles.

Ahora, con las dos terminales a la vista, ejecutar la celda E4b.

In [9]:
encabezado("E4b")
estado, miembros = estado_grupo()
if estado != "Stable" or miembros != 2:
    print("ATENCION: E4b exige los dos consumidores activos y el grupo estable (estado actual: {}, miembros: {}).".format(estado, miembros))
    print("Completar los pasos 1 a 3 y ejecutar de nuevo esta celda. No se produjo ningun mensaje.")
elif not ya_ejecutada("E4b"):
    claves_e4 = ["sensor-{}".format(random.randint(100, 999)) for _ in range(12)]
    with open(T10 + "/e4b.txt", "w") as f:
        for i, clave in enumerate(claves_e4, 1):
            f.write("{}:lectura-e4-{:02d}\n".format(clave, i))
    !{KB}/kafka-console-producer.sh --bootstrap-server {BS} --topic {TOPIC} --property parse.key=true --property key.separator=: < {T10}/e4b.txt
    marcar("E4b", "12 con clave")
    print("12 mensajes producidos. Contar cuantas lineas nuevas aparecieron en cada terminal.")

E4b | 2026-10-28 17:13:30 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
12 mensajes producidos. Contar cuantas lineas nuevas aparecieron en cada terminal.


Ahora detener el consumidor de la terminal B con `Ctrl + C` (paso 5), esperar unos segundos y ejecutar la celda E4c.

In [10]:
encabezado("E4c")
!{KB}/kafka-consumer-groups.sh --bootstrap-server {BS} --describe --group {GRUPO} --members --verbose

E4c | 2026-10-28 17:16:43 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw

GROUP           CONSUMER-ID                                           HOST            CLIENT-ID        #PARTITIONS     ASSIGNMENT
grupo10         console-consumer-05194e76-9a68-434d-8dc0-814aab3d94db /192.168.56.101 console-consumer 3               (0,1,2)


Por último, detener el consumidor de la terminal A con `Ctrl + C` (paso 7).

**Registrar en la hoja (fila E4):**

- De la celda E4a: para cada miembro, `HOST`, `#PARTITIONS` y particiones asignadas (`ASSIGNMENT`).
- Del paso 4: **cuántas** líneas nuevas aparecieron en la terminal A y cuántas en la terminal B (el número, no el contenido).
- Del paso 5: el número N que imprimió la terminal B en `Processed a total of N messages`.
- De la celda E4c: `#PARTITIONS` del miembro que quedó.
- Del paso 7: el número N que imprimió la terminal A.
- La suma de los dos totales (A + B). Debe poder explicarse por qué da exactamente esa cifra.

*No requiere captura adicional:* C3 ya se tomó en el paso 3.

---
## Ejecución 5 — Offsets confirmados y lag

**Las dos terminales deben tener los consumidores detenidos antes de ejecutar esta parte.**

Cada grupo **confirma** hasta dónde ha leído en cada partición, y Kafka guarda esos offsets confirmados en un topic interno, `__consumer_offsets`. La consulta `kafka-consumer-groups.sh --describe` muestra, por partición:

- `CURRENT-OFFSET`: offset confirmado por el grupo.
- `LOG-END-OFFSET`: offset que recibirá el próximo mensaje que se escriba en la partición.
- `LAG`: la diferencia entre los dos, es decir, los mensajes que existen y que el grupo todavía no ha leído.

**E5a.** Con el grupo sin miembros activos, se producen **M** mensajes con clave, donde M es un número elegido al azar por la celda. Después se describe el grupo. El aviso `Consumer group 'grupo10' has no active members` es esperado: el grupo existe y conserva sus offsets aunque ningún consumidor esté conectado.

**E5b.** Un consumidor del mismo grupo lee exactamente M mensajes y termina. Después se describe el grupo de nuevo.

In [11]:
encabezado("E5a")
estado, miembros = estado_grupo()
if estado == "NO_EXISTE" or not os.path.exists(T10 + "/E4b.hecho"):
    print("ATENCION: la Ejecucion 4 no se ha completado. Los consumidores de las terminales A y B crean el grupo,")
    print("y la celda E4b produce los 12 mensajes. Completar la Ejecucion 4 antes de esta celda. No se produjo ningun mensaje.")
elif miembros > 0:
    print("ATENCION: el grupo grupo10 todavia tiene {} consumidor(es) activo(s) (estado: {}).".format(miembros, estado))
    print("Detenerlos con Ctrl + C en las terminales A y B, esperar unos segundos y ejecutar de nuevo esta celda.")
elif not ya_ejecutada("E5a"):
    M = random.randint(25, 45)
    print("M =", M, "mensajes producidos con el grupo detenido")
    with open(T10 + "/e5a.txt", "w") as f:
        for i in range(1, M + 1):
            f.write("sensor-{}:lectura-e5-{:02d}\n".format(random.randint(100, 999), i))
    !{KB}/kafka-console-producer.sh --bootstrap-server {BS} --topic {TOPIC} --property parse.key=true --property key.separator=: < {T10}/e5a.txt
    marcar("E5a", "M = {}".format(M))
    time.sleep(2)
    print()
    !{KB}/kafka-consumer-groups.sh --bootstrap-server {BS} --describe --group {GRUPO}

E5a | 2026-10-28 17:21:24 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
M = 44 mensajes producidos con el grupo detenido


Consumer group 'grupo10' has no active members.

GROUP           TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
grupo10         taller10        1          30              39              9               -               -               -
grupo10         taller10        2          33              50              17              -               -               -
grupo10         taller10        0          19              37              18              -               -               -

**Registrar en la hoja (fila E5a):** valor de M · para cada partición, `CURRENT-OFFSET`, `LOG-END-OFFSET` y `LAG` · suma de la columna `LAG` · suma de la columna `LOG-END-OFFSET`.

**Captura C4:** la salida completa de esta celda, con el valor de M y la tabla del grupo a la vista.

*Comprobación obligatoria:* la suma de `LAG` debe coincidir con M, y la suma de `LOG-END-OFFSET` debe coincidir con el total de mensajes producidos en todo el taller (E3, E4b y E5a). Si alguna no coincide, informar a la docente antes de continuar.

In [12]:
encabezado("E5b")
if not os.path.exists(T10 + "/E5a.hecho"):
    print("ATENCION: la celda E5a todavia no ha producido los M mensajes. Ejecutarla primero.")
elif ya_ejecutada("E5b"):
    !{KB}/kafka-consumer-groups.sh --bootstrap-server {BS} --describe --group {GRUPO}
else:
    with open(T10 + "/E5a.hecho") as f:
        M = int(f.read().split("M = ")[1])   # M se lee de la marca de E5a
    print("Se leen exactamente M =", M, "mensajes con el grupo", GRUPO)
    !{KB}/kafka-console-consumer.sh --bootstrap-server {BS} --topic {TOPIC} --group {GRUPO} --max-messages {M} --timeout-ms 30000 --property print.partition=true --property print.offset=true
    marcar("E5b", "M = {}".format(M))
    time.sleep(2)
    print()
    !{KB}/kafka-consumer-groups.sh --bootstrap-server {BS} --describe --group {GRUPO}

E5b | 2026-10-28 17:30:00 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw
Se leen exactamente M = 44 mensajes con el grupo grupo10
Partition:0	Offset:19	lectura-e5-02
Partition:0	Offset:20	lectura-e5-04
Partition:0	Offset:21	lectura-e5-10
Partition:0	Offset:22	lectura-e5-14
Partition:0	Offset:23	lectura-e5-15
Partition:0	Offset:24	lectura-e5-19
Partition:0	Offset:25	lectura-e5-24
Partition:0	Offset:26	lectura-e5-26
Partition:0	Offset:27	lectura-e5-28
Partition:0	Offset:28	lectura-e5-29
Partition:0	Offset:29	lectura-e5-30
Partition:0	Offset:30	lectura-e5-35
Partition:0	Offset:31	lectura-e5-36
Partition:0	Offset:32	lectura-e5-37
Partition:0	Offset:33	lectura-e5-39
Partition:0	Offset:34	lectura-e5-40
Partition:0	Offset:35	lectura-e5-42
Partition:0	Offset:36	lectura-e5-43
Partition:1	Offset:30	lectura-e5-05
Partition:1	Offset:31	lectura-e5-07
Partition:1	Offset:32	lectura-e5-17
Partition:1	Offset:33	lectura-e5-18
Partition:1	Offset:34	lectura-e5-20
Partition:1	Offset:35	lectura-e5-21
Partition:1	Offse

**Registrar en la hoja (fila E5b):** total que imprimió el consumidor (`Processed a total of …`) · `LAG` de cada partición después de la lectura.

*No requiere captura.* Durante la Fase 2 se tomará una captura nueva de este mismo estado del grupo, y una de las preguntas pide compararla con C4.

---
## Cierre de la Fase 1

Antes del corte, verificar que:

1. La hoja de registro está completa, filas E1 a E5b.
2. El archivo de Word contiene las cuatro capturas, numeradas C1 a C4, y todas muestran el mismo `ClusterId`.
3. El archivo se convirtió a PDF y se entregó por Microsoft Teams.

**Durante la Fase 2:**

- **No detener Kafka.** El clúster debe seguir activo: una pregunta exige una captura nueva tomada durante la fase.
- **Cerrar la pestaña de este notebook.**
- En la terminal **solo** puede ejecutarse el script de consulta, con una de estas cinco opciones:

| Comando | Qué muestra |
|---|---|
| `bash ~/t10/consulta.sh topic` | Descripción del topic `taller10` |
| `bash ~/t10/consulta.sh grupo` | Offsets y lag del grupo `grupo10` |
| `bash ~/t10/consulta.sh miembros` | Miembros activos del grupo |
| `bash ~/t10/consulta.sh offsets` | Último offset de cada partición |
| `bash ~/t10/consulta.sh quorum` | Estado del quórum de controllers |

- El cuestionario se responde en **Microsoft Forms**. La captura nueva se carga en el mismo formulario.

**Material permitido en la Fase 2:** la terminal, únicamente para `consulta.sh`; la hoja de registro diligenciada; el archivo de capturas entregado en el corte, únicamente para consulta; los notebooks de sesiones anteriores en la versión publicada por la docente en Microsoft Teams; los apuntes propios manuscritos.

**No permitido:** herramientas de inteligencia artificial; cualquier otro comando en la terminal; copias propias o editadas de los notebooks; apuntes digitales; el notebook de este taller, que debe permanecer cerrado.

In [13]:
!bash ~/t10/consulta.sh grupo

consulta.sh grupo | 2026-10-28 17:33:38 | ClusterId: Rxkee_6WRYaGnf3eVYjxNw

Consumer group 'grupo10' has no active members.

GROUP           TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
grupo10         taller10        1          39              39              0               -               -               -
grupo10         taller10        2          50              50              0               -               -               -
grupo10         taller10        0          37              37              0               -               -               -

In [15]:
# ATENCION: esta linea debe permanecer comentada durante todo el taller.
# Ejecutarla detiene los tres brokers y deja sin respaldo la captura nueva de la Fase 2.
!bash /home/hadoop/kafka_cluster.sh detener

Deteniendo los tres brokers...
  maestro (192.168.56.101)  detenido
  nodo2   (192.168.56.102)  detenido
  nodo3   (192.168.56.103)  detenido
